# Dynamic Factor Model (Pipeline B)

**Model**: DFM via `nowcastDFM::dfm`
**Target**: `gdpc1`
**Variables**: Cat3 monthly-frequency only + gdpc1 (~31 vars). Quarterly Cat3 excluded.
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: YES (factor loadings depend on variance)
**HP tuning**: NO (1 global + per-block factors per Bok et al. 2018)
**Estimation**: EM algorithm, one fit (not rolling). Kalman filter handles NaN.
**Prediction window**: 30-year rolling window passed to predict_dfm (memory bound).
**Note**: `nowcastDFM` works with data.frame, not tibble. Block columns read dynamically from meta_data.csv.
**Quarterly-var exclusion**: nowcastDFM represents each quarterly variable with 5 Kalman states.
With nQ=23 quarterly Cat3 vars, `eye((5*nQ)^2)` = 115×115 identity → 2.5 GB OOM in init_conds.
Fix: only the target (gdpc1, nQ=1) is quarterly; all other Cat3 quarterly vars excluded from DFM.
COVID dummies also excluded (zero variance in training → scale() → NaN → EM fails).

In [ ]:
library(tidyverse)
library(nowcastDFM)

source("../data/helpers.R")

metadata <- read_csv("../data/meta_data.csv")

cat3_features <- c(
  "a014re1q156nbea","acogno","ahetpix","amdmuox","andenox","awotman",
  "busloans","ce16ov","ces1021000001","ces2000000008","ces9091000001",
  "ces9092000001","clf16ov","compapff","cusr0000sas","ddurrg3m086sbea",
  "dhlcrg3q086sbea","difsrg3q086sbea","dodgrg3q086sbea","dongrg3q086sbea",
  "dspic96","expgsc1","fpix","gcec1","gpdic1","houstne","housts",
  "hwiuratio","hwiuratiox","invest","ipdcongd","liabpix","lns13023705",
  "m2sl","manemp","mortg10yrx","nonrevsl","ophpbs","outbs","outnfb",
  "permitw","realln","slcex","spcs10rsa","tlbsnncbbdix","uemp15t26",
  "uemp27ov","uemplt5","ulcbs","ulcnfb","unrate","usgovt","usserv",
  "covid_2020q2","covid_2020q3","covid_2020q4"
)
cat3_features <- tolower(cat3_features)

# Identify and exclude quarterly vars and COVID dummies.
# nowcastDFM needs 5 Kalman states per quarterly variable; with nQ=23 vars,
# eye((5*nQ)^2) = 115x115 identity exhausts 2.5 GB RAM in init_conds.
# Solution: only gdpc1 (the target) is quarterly (nQ=1 -> eye(25) -> trivial).
covid_dummies   <- c("covid_2020q2", "covid_2020q3", "covid_2020q4")
quarterly_cat3  <- metadata %>%
    dplyr::filter(series %in% cat3_features, freq == "q", series != "gdpc1") %>%
    pull(series)

cat3_for_dfm <- cat3_features[
    !cat3_features %in% covid_dummies &
    !cat3_features %in% quarterly_cat3
]

data <- read_csv("../data/data_tf_monthly.csv") %>% arrange(date)

target_variable  <- "gdpc1"
train_start_date <- "1959-01-01"
test_start_date  <- "2017-01-01"
test_end_date    <- "2025-12-01"

# End-of-quarter dates (Mar/Jun/Sep/Dec): gdpc1 is only observed at quarter end.
# seq("2017-01-01", ...) gives Jan/Apr/Jul/Oct which return NA from predict_dfm.
test_dates <- seq(as.Date("2017-03-01"), as.Date(test_end_date), by = "3 months")

test <- data %>%
    dplyr::filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>%
    data.frame()

for (col in colnames(test)) {
    if (is.numeric(test[, col]) && sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

print(paste("DFM variables: monthly Cat3 + gdpc1 =", length(cat3_for_dfm) + 1,
            "| Excluded:", length(quarterly_cat3), "quarterly +", length(covid_dummies), "COVID"))

In [ ]:
# Patch nowcastDFM:::init_conds — cov(res[, idx_iM]) fails when a block has exactly
# 1 monthly variable because R drops dimensions, returning a vector instead of a matrix.
# Fix: wrap with as.matrix(..., drop=FALSE) to preserve 2-D shape.
local({
  fn <- nowcastDFM:::init_conds
  b  <- body(fn)
  # Replace cov(res[, idx_iM]) with cov(as.matrix(res[, idx_iM, drop=FALSE]))
  replace_cov <- function(node) {
    if (is.call(node)) {
      txt <- deparse(node)
      if (identical(txt, "cov(res[, idx_iM])")) {
        return(quote(cov(as.matrix(res[, idx_iM, drop = FALSE]))))
      }
      return(as.call(lapply(node, replace_cov)))
    }
    node
  }
  body(fn) <- replace_cov(b)
  assignInNamespace("init_conds", fn, ns = "nowcastDFM")
  cat("nowcastDFM:::init_conds patched (drop=FALSE for single-variable blocks)\n")
})

In [ ]:
# Build DFM variable list: monthly Cat3 + gdpc1 only
# col_names_dfm preserves the column order from data_tf_monthly.csv
col_names_dfm <- colnames(test)[2:ncol(test)]
col_names_dfm <- col_names_dfm[col_names_dfm %in% c("gdpc1", cat3_for_dfm)]

# Build blocks from metadata (block structure from Bok et al. 2018)
blocks <- metadata %>%
    dplyr::filter(series %in% col_names_dfm) %>%
    dplyr::filter(series %in% colnames(test))
blocks <- blocks[match(col_names_dfm, blocks$series), ]
blocks <- blocks %>%
    select(starts_with("block_")) %>%
    select_if(~ sum(.) > 0) %>%
    data.frame()
print(paste("Blocks:", ncol(blocks), "block columns for", nrow(blocks), "variables"))

# Fit DFM once on training data (1959-2007) — static model, not rolling
train_cols <- c("date", col_names_dfm)
train <- test %>%
    dplyr::filter(date <= as.Date("2007-12-31")) %>%
    dplyr::filter(date >= as.Date(train_start_date)) %>%
    data.frame()
train <- train[, train_cols]
train$date <- as.character(train$date)
output_dfm <- dfm(data = train, blocks = blocks, max_iter = 500)
cat("DFM fitted on", nrow(train), "months x", nrow(blocks), "variables\n")

In [ ]:
# Rolling prediction loop
vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) {
    pred_dict[, lag_name] <- NA
}

# Cap window passed to predict_dfm to 30 years (360 months) to bound Kalman smoother memory.
# The model parameters are fixed from training; we only need enough history for the Kalman
# filter to converge, and 30 years far exceeds any GDP cycle length.
WINDOW_MONTHS <- 360

for (i in 1:length(test_dates)) {
    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data <- lagged_data[, train_cols]  # match DFM training columns
        # Keep date as Date (NOT character): predict_dfm's add_month loop assigns as.Date()
        # back to the date column; if column is character, Date coerces to integer "17258"
        # causing substr() to return "" -> month=NA -> if(month==12) crashes.
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA

        # Trim to rolling window
        if (nrow(lagged_data) > WINDOW_MONTHS) {
            lagged_data <- tail(lagged_data, WINDOW_MONTHS)
        }

        prediction <- tryCatch({
            predict_dfm(lagged_data, output_dfm) %>%
                dplyr::filter(date == test_dates[i]) %>%
                select(!!target_variable) %>%
                pull()
        }, error = function(e) NA)
        pred_dict[pred_dict$date == test_dates[i], lag_name] <- prediction
    }
    if (i %% 8 == 0) print(paste(i, "/", length(test_dates)))
}

# Extract actuals at quarter-end months matching test_dates
actuals <- test %>%
    dplyr::filter(date %in% as.Date(test_dates)) %>%
    select(!!target_variable) %>%
    pull()

In [ ]:
# Save predictions
dir.create("../predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(
        date = test_dates,
        actual = actuals,
        prediction = pred_dict[, lag_name]
    )
    write.csv(df_out, paste0("../predictions/dfm_", lag_name, ".csv"), row.names = FALSE)
}

# Evaluation
panels <- list(
    "Pre-COVID  (2017-2019)" = c("2017-03-01", "2019-12-31"),
    "COVID      (2020-2021)" = c("2020-06-01", "2021-12-31"),
    "Post-COVID (2022-2025)" = c("2022-03-01", "2025-12-31"),
    "Full test  (2017-2025)" = c("2017-03-01", "2025-12-31")
)

rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)

for (pn in names(panels)) {
    rng <- panels[[pn]]
    m <- test_dates >= rng[1] & test_dates <= rng[2]
    cat("--- ", pn, " ---\n")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits=6),
            "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits=6), "\n")
    }
}